# Projeto TraficGenius: Storytelling Estatístico e Modelagem Preditiva de Severidade
Este notebook apresenta a análise científica e o storytelling estatístico completo para a modelagem preditiva de severidade de acidentes rodoviários (Leve/Moderado vs. Grave/Fatal), utilizando o modelo final unificado com **Flag de País** (contendo a feature `pais_US` para calibração conjunta da base americana e brasileira).

O objetivo é expor com rigor acadêmico a aderência dos dados às premissas clássicas, o comportamento exploratório multivariado e a capacidade discriminante e interpretativa da arquitetura preditiva desenvolvida.

---

## Estrutura do Notebook:
1. **Etapa 1: Ingestão de Dados e Preparação**: Carregamento da base tratada e codificação dummy.
2. **Etapa 2: Diagnóstico de Outliers**: Z-Score univariado e Distância de Mahalanobis ($D^2$) multivariada.
3. **Etapa 3: Regressão Linear Auxiliar e Diagnósticos de Premissas Clássicas**: Extração de resíduos e execução de testes formais de normalidade (JB, Shapiro, KS), homocedasticidade (Breusch-Pagan, Levene), independência (Durbin-Watson, Runs Test), colinearidade (VIF, Condition Index) e especificação (Ramsey RESET).
4. **Etapa 4: Visualizações de Premissas**: Gráficos NPP Q-Q plot, Resíduos vs. Ajustados, ACF Correlograma e VIF Bar Chart.
5. **Etapa 5: Modelagem Preditiva com Classificadores e Stacking**: Carregamento de modelos, otimização de limiares de decisão (Threshold Tuning) e avaliação de métricas.
6. **Etapa 6: Avaliação de Desempenho Avançado e Explicações SHAP**: Geração das curvas ROC, PR, Calibração, Lift & Ganho, projeção espacial 2D t-SNE e SHAP values do XGBoost.


In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, linear_reset
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score, f1_score,
    accuracy_score, balanced_accuracy_score, log_loss, roc_curve, auc, precision_recall_curve
)
from sklearn.manifold import TSNE
import shap

warnings.filterwarnings('ignore')

# Configuração de estilo visual neon escuro do TraficGenius para os gráficos
def apply_chart_style():
    plt.style.use('dark_background')
    plt.rcParams['figure.facecolor'] = '#121214'
    plt.rcParams['axes.facecolor'] = '#1a1a1e'
    plt.rcParams['grid.color'] = '#2d2d34'
    plt.rcParams['font.sans-serif'] = 'sans-serif'
    plt.rcParams['font.family'] = 'sans-serif'

apply_chart_style()
print("[INFO] Bibliotecas importadas e estilo de gráficos configurado com sucesso!")


## Etapa 1: Ingestão de Dados e Preparação
Nesta etapa, carregamos a amostra limpa do conjunto de dados binário, criamos a feature de controle nacional `pais_US` (1 para EUA, 0 para Brasil) e aplicamos o One-Hot Encoding nas variáveis categóricas de rodovia dominante.


In [ ]:
# Carregamento dos dados
input_file = "dataset/dataset_amostra_limpa_binaria.parquet"
df = pd.read_parquet(input_file)
print(f"Dataset original carregado. Shape: {df.shape}")

# Criação da flag de país e remoção de campos de controle geográfico/temporal absoluto
df['pais_US'] = (df['pais'] == 'US').astype(int)
df_clean = df.drop(columns=['data_inversa', 'pais'])

# One-hot encoding
cat_cols = ['rodovia_dominante_res11', 'rodovia_dominante_res10', 'rodovia_dominante_res9']
df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)

target_col = 'severidade_binaria'
feature_cols = [col for col in df_encoded.columns if col != target_col]

print(f"Dataset pronto para uso após dummy encoding. Shape: {df_encoded.shape}")
df_encoded.head()


## Etapa 2: Diagnóstico de Outliers (Univariados e Multivariados)
Utilizamos duas abordagens:
1. **Z-Score e Z-Score Modificado (baseado em MAD)**: Para identificar anomalias univariadas nas features contínuas físicas.
2. **Distância de Mahalanobis ($D^2$)**: Para computar anomalias multivariadas com base na distribuição Qui-Quadrado ($\chi^2$) a $99.9\%$ de confiança.


In [ ]:
# Amostragem representativa de 50.000 registros para os cálculos computacionais
df_sample_large = df.sample(n=min(50000, len(df)), random_state=42).copy()
df_sample_large['pais_US'] = (df_sample_large['pais'] == 'US').astype(int)

# 1. Outliers Univariados
print("--- 1. Análise de Outliers Univariados ---")
for col in ['velocidade_media', 'temperatura_celsius', 'precipitacao_milimetros']:
    series = df_sample_large[col]
    z_score = np.abs(stats.zscore(series))
    outlier_z = np.sum(z_score > 3.5) / len(series)
    
    median = series.median()
    mad = np.median(np.abs(series - median))
    if mad == 0: mad = 1e-5
    mod_z = 0.6745 * (series - median) / mad
    outlier_mod_z = np.sum(np.abs(mod_z) > 3.5) / len(series)
    
    print(f"Feature: {col}")
    print(f"  Outliers Z-Score (>3.5): {outlier_z*100:.3f}%")
    print(f"  Outliers Mod. Z-Score (MAD >3.5): {outlier_mod_z*100:.3f}%")

# 2. Outliers Multivariados (Mahalanobis D^2)
print("\n--- 2. Análise de Outliers Multivariados (Mahalanobis) ---")
cols_cont = [
    'quantidade_cruzamentos', 'quantidade_semaforos', 'velocidade_media',
    'quantidade_faixas_media', 'curvatura_acumulada', 'desvio_maximo_curvatura',
    'quantidade_curvas_acentuadas', 'quantidade_rotatorias', 'quantidade_pontes',
    'comprimento_pontes_metros', 'quantidade_tuneis', 'comprimento_tuneis_metros',
    'quantidade_postos_combustivel', 'quantidade_restaurantes', 'quantidade_escolas',
    'quantidade_hospitais', 'quantidade_rodovias_distintas', 'total_curvas_acentuadas',
    'quantidade_locais_interesse', 'area_urbana_m2', 'area_rural_m2',
    'temperatura_celsius', 'ponto_orvalho_celsius', 'pressao_hpa',
    'velocidade_vento_u', 'velocidade_vento_v', 'cobertura_nuvens_percentual',
    'precipitacao_milimetros'
]
cols_cont = [c for c in cols_cont if c in df.columns]
df_cont = df_sample_large[cols_cont]

mean = df_cont.mean(axis=0)
cov = df_cont.cov()
inv_cov = np.linalg.pinv(cov)
diff = df_cont - mean
d2 = np.sum(np.dot(diff, inv_cov) * diff, axis=1)

crit_val = stats.chi2.ppf(0.999, df=len(cols_cont))
mahalanobis_outliers = np.sum(d2 > crit_val) / len(d2)

print(f"Graus de Liberdade (DF): {len(cols_cont)}")
print(f"Valor Crítico de Chi-Square (p < 0.001): {crit_val:.4f}")
print(f"Distância Máxima de Mahalanobis encontrada: {np.max(d2):.4f}")
print(f"Proporção de Outliers Multivariados: {mahalanobis_outliers*100:.3f}%")


## Etapa 3: Regressão Linear Auxiliar e Diagnósticos de Premissas Clássicas
Para realizar os testes clássicos de resíduos da econometria, ajustamos um modelo de Mínimos Quadrados Ordinários (OLS) auxiliar nos preditores (incluindo `pais_US`) e extraímos os resíduos. Os seguintes testes formais são calculados:
1. **Normalidade**: Coeficientes de Assimetria e Curtose, testes formais de Jarque-Bera (JB), Shapiro-Wilk (SW) e Kolmogorov-Smirnov (KS).
2. **Homocedasticidade**: Teste de Breusch-Pagan-Godfrey (BPG) e Teste de Levene para igualdade de variância da velocidade média entre as duas classes.
3. **Independência dos Erros**: Estatística de Durbin-Watson e Runs Test (Teste das Carreiras).
4. **Multicolinearidade**: Fatores de Inflação da Variância (VIF) e Número de Condição (Condition Index).
5. **Especificação**: Teste RESET de Ramsey para termos não-lineares omitidos.


In [ ]:
# 1. Regressão Auxiliar em amostra pequena de 5.000 para testes que requerem baixa dimensionalidade
df_sample_small = df.sample(n=5000, random_state=42).copy()
df_sample_small['pais_US'] = (df_sample_small['pais'] == 'US').astype(int)
df_small_encoded = pd.get_dummies(df_sample_small, columns=cat_cols, drop_first=True)

# Alinha colunas
for c in feature_cols:
    if c not in df_small_encoded.columns:
        df_small_encoded[c] = 0

X_reg = sm.add_constant(df_small_encoded[feature_cols].astype(float))
y_reg = df_small_encoded[target_col].astype(float)

# Ajuste OLS
ols_model = sm.OLS(y_reg, X_reg).fit()
residuals = ols_model.resid
fitted = ols_model.fittedvalues

# Helper para Runs Test
def run_runs_test(resids):
    median_val = np.median(resids)
    runs = np.diff(resids > median_val) != 0
    num_runs = np.sum(runs) + 1
    n1 = np.sum(resids > median_val)
    n2 = np.sum(resids <= median_val)
    expected_runs = ((2.0 * n1 * n2) / (n1 + n2)) + 1.0
    var_runs = (2.0 * n1 * n2 * (2.0 * n1 * n2 - n1 - n2)) / (((n1 + n2) ** 2) * (n1 + n2 - 1.0))
    z_stat = (num_runs - expected_runs) / np.sqrt(var_runs)
    p_val = 2.0 * (1.0 - stats.norm.cdf(np.abs(z_stat)))
    return z_stat, p_val

print("=== 1. Testes de Aderência à Normalidade ===")
skewness = stats.skew(residuals)
kurtosis = stats.kurtosis(residuals)
jb_stat, jb_pval = stats.jarque_bera(residuals)
sw_stat, sw_pval = stats.shapiro(residuals)
std_residuals = (residuals - residuals.mean()) / residuals.std()
ks_stat, ks_pval = stats.kstest(std_residuals, 'norm')
print(f"Assimetria (Skewness): {skewness:.4f}")
print(f"Curtose (Kurtosis): {kurtosis:.4f}")
print(f"Jarque-Bera: Estatística={jb_stat:.2f}, P-Valor={jb_pval}")
print(f"Shapiro-Wilk: Estatística={sw_stat:.4f}, P-Valor={sw_pval}")
print(f"Kolmogorov-Smirnov: Estatística={ks_stat:.4f}, P-Valor={ks_pval}")

print("\n=== 2. Testes de Homocedasticidade ===")
bp_lm, bp_lm_p, bp_f, bp_f_p = het_breuschpagan(residuals, X_reg)
grp0 = df_sample_small[df_sample_small['severidade_binaria'] == 0]['velocidade_media']
grp1 = df_sample_small[df_sample_small['severidade_binaria'] == 1]['velocidade_media']
levene_stat, levene_pval = stats.levene(grp0, grp1)
print(f"Breusch-Pagan LM Stat: {bp_lm:.2f}, P-Valor={bp_lm_p}")
print(f"Levene (velocidade_media por classe): Estatística={levene_stat:.4f}, P-Valor={levene_pval}")

print("\n=== 3. Testes de Independência (Autocorrelação) ===")
dw_stat = sm.stats.durbin_watson(residuals)
runs_z, runs_pval = run_runs_test(residuals.to_numpy())
print(f"Durbin-Watson: {dw_stat:.4f}")
print(f"Runs Test: Z-Stat={runs_z:.4f}, P-Valor={runs_pval}")

print("\n=== 4. Testes de Multicolinearidade ===")
X_cont = df_sample_small[cols_cont].astype(float)
X_cont = sm.add_constant(X_cont)
vifs = []
for i in range(X_cont.shape[1]):
    col_name = X_cont.columns[i]
    val = variance_inflation_factor(X_cont.values, i)
    vifs.append({'feature': col_name, 'vif': val})
vif_df = pd.DataFrame(vifs)
max_vif = vif_df[vif_df['feature'] != 'const']['vif'].max()

# Condition Index
X_scaled = (X_cont.iloc[:, 1:] - X_cont.iloc[:, 1:].mean()) / X_cont.iloc[:, 1:].std()
X_scaled = sm.add_constant(X_scaled).dropna(axis=1, how='all')
U, s, Vt = np.linalg.svd(X_scaled)
s_max = np.max(s)
condition_indices = [s_max / val for val in s if val > 0]
print(f"VIF Máximo (excluindo intercepto): {max_vif:.4f}")
print(f"Condition Index Máximo: {np.max(condition_indices):.4f}")

print("\n=== 5. Teste de Especificação (Ramsey RESET) ===")
try:
    reset_test = linear_reset(ols_model, power=3, test_type='fitted')
    print(f"Ramsey RESET F-Stat: {reset_test.statistic:.4f}, P-Valor={reset_test.pvalue}")
except Exception as e:
    print(f"Ramsey RESET Erro: {e}")


## Etapa 4: Visualizações de Premissas Estatísticas
Geramos os gráficos para diagnosticar visualmente os resíduos do modelo OLS:
1. **Normal Q-Q Plot (NPP)**: Verifica desvios de cauda e normalidade.
2. **Resíduos vs. Valores Ajustados**: Avalia a dispersão da variância residual (Homocedasticidade).
3. **Correlograma ACF**: Plota os coeficientes de autocorrelação serial dos erros.
4. **Bar Chart de VIF**: Identifica features colineares de forma visual.


In [ ]:
# 1. Normal Q-Q Plot
fig, ax = plt.subplots(figsize=(7, 6))
sm.qqplot(residuals, line='45', fit=True, ax=ax)
ax.get_lines()[0].set_color('#ff007f')
ax.get_lines()[0].set_markersize(4)
ax.get_lines()[1].set_color('#00f0ff')
ax.get_lines()[1].set_linewidth(2)
plt.title('Gráfico de Probabilidade Normal (NPP Q-Q Plot) dos Resíduos')
plt.show()

# 2. Resíduos vs. Valores Ajustados (Homocedasticidade)
plt.figure(figsize=(8, 5))
plt.scatter(fitted, residuals, alpha=0.3, color='#00f0ff', s=10)
plt.axhline(y=0, color='#ff007f', linestyle='--', lw=2)
plt.title('Homocedasticidade: Resíduos vs. Valores Ajustados')
plt.xlabel('Valores Ajustados (Previsão Linear)')
plt.ylabel('Resíduos do Modelo')
plt.grid(True, linestyle=':', alpha=0.4)
plt.show()

# 3. Correlograma ACF
fig, ax = plt.subplots(figsize=(8, 5))
plot_acf(residuals, lags=20, ax=ax, color='#00f0ff', vlines_kwargs={"colors": '#ff007f'})
plt.title('Correlograma de Autocorrelação ACF dos Erros')
plt.grid(True, linestyle=':', alpha=0.4)
plt.show()

# 4. VIF Bar Chart
vif_clean = vif_df[vif_df['feature'] != 'const'].sort_values(by='vif', ascending=True)
plt.figure(figsize=(10, 6))
colors = ['#ff007f' if v > 10 else '#00f0ff' for v in vif_clean['vif']]
plt.barh(vif_clean['feature'], vif_clean['vif'], color=colors, height=0.6)
plt.axvline(x=10.0, color='r', linestyle='--', label='Limite Tolerável (VIF = 10)')
plt.title('Fator de Inflação da Variância (VIF) dos Preditores')
plt.xlabel('VIF')
plt.legend()
plt.tight_layout()
plt.show()


## Etapa 5: Modelagem Preditiva com Classificadores e Stacking
Nesta etapa, carregamos os modelos pré-treinados finais com a Flag de País (Regressão Logística, LDA, Random Forest, Rede Neural MLP e XGBoost) e o Stacking Ensemble (meta-aprendedor Logit).
Otimizamos os limiares de decisão de cada modelo no meta-treino para maximizar o F1-Score da classe grave e reportamos a tabela comparativa no conjunto de teste final de 2,35 milhões de registros.


In [ ]:
# Divisão treino/teste estratificada para recriar partições e scaler
X = df_encoded[feature_cols]
y = df_encoded[target_col]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

scaler = joblib.load("dataset/scaler_flag_pais.joblib")
X_test_scaled = scaler.transform(X_test)

# Carrega modelos salvos
models = {}
models['Logistic Regression'] = joblib.load("dataset/model_logit_flag_pais.joblib")
models['LDA'] = joblib.load("dataset/model_lda_flag_pais.joblib")
models['Random Forest'] = joblib.load("dataset/model_rf_flag_pais.joblib")
models['XGBoost'] = joblib.load("dataset/model_xgb_flag_pais.joblib")
models['Stacking Ensemble'] = joblib.load("dataset/model_stacking_flag_pais.joblib")

# Carrega MLP
TENSORFLOW_AVAILABLE = False
try:
    import tensorflow as tf
    models['Neural Network (MLP)'] = tf.keras.models.load_model("dataset/model_mlp_flag_pais.h5")
    TENSORFLOW_AVAILABLE = True
    print("[INFO] TensorFlow carregado com sucesso. Rede Neural carregada do formato H5.")
except Exception:
    if os.path.exists("dataset/model_mlp_flag_pais.joblib"):
        models['Neural Network (MLP)'] = joblib.load("dataset/model_mlp_flag_pais.joblib")
        print("[INFO] Rede Neural carregada via MLPClassifier (fallback joblib).")

# Carrega limiares ótimos otimizados no meta-treino
with open("dataset/limiares_decisao_flag_pais.json", "r") as f:
    optimal_thresholds = json.load(f)

print("\nLimiares de Decisão Ótimos Otimizados:")
for k, v in optimal_thresholds.items():
    print(f" - {k:22}: {v:.2f}")

# Carrega e exibe a tabela de métricas comparativas no conjunto de teste
metrics_df = pd.read_csv("dataset/comparativo_modelos_flag_pais.csv")
print("\nTabela Comparativa de Métricas de Avaliação no Teste:")
metrics_df


## Etapa 6: Avaliação de Desempenho Avançado e Explicações SHAP
Calculamos as probabilidades previstas no conjunto de teste e geramos os seguintes plots avançados:
1. **Curvas ROC (AUC)**: Capacidade de discriminação geral.
2. **Precision-Recall (PR)**: Precisão preditiva sobre a classe de acidentes graves sob diferentes taxas de recall.
3. **Diagrama de Calibração (Reliability Diagram)**: Calibração de probabilidade dos modelos.
4. **Curvas de Lift e Ganho Acumulado**: Utilidade comercial de priorização.
5. **Projeção t-SNE 2D**: Separação espacial no espaço latente.
6. **Explicações Globais SHAP**: Importância das variáveis baseada na contribuição marginal SHAP no XGBoost.


In [ ]:
# Seleciona amostra de teste para agilidade gráfica (15000 registros)
np.random.seed(42)
sample_indices = np.random.choice(len(X_test), size=min(15000, len(X_test)), replace=False)
X_sample_df = X_test.iloc[sample_indices]
X_sample_scaled = X_test_scaled[sample_indices]
y_sample = y_test.iloc[sample_indices]

probs = {}
meta_preds_sample = {}
for name, model in models.items():
    if name == 'Stacking Ensemble': continue
    if name == 'Neural Network (MLP)' and TENSORFLOW_AVAILABLE:
        p = model.predict(X_sample_scaled).ravel()
    else:
        p = model.predict_proba(X_sample_scaled)[:, 1]
    probs[name] = p
    meta_preds_sample[name] = p

X_meta_sample = pd.DataFrame(meta_preds_sample)
expected_cols = ['Logistic Regression', 'LDA', 'Random Forest', 'Neural Network (MLP)', 'XGBoost']
X_meta_sample = X_meta_sample[expected_cols]
probs['Stacking Ensemble'] = models['Stacking Ensemble'].predict_proba(X_meta_sample)[:, 1]

# A. Curvas ROC
plt.figure(figsize=(8, 7))
for name, p in probs.items():
    fpr, tpr, _ = roc_curve(y_sample, p)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='#e2e2e9', linestyle='--', label='Baseline Aleatório')
plt.title('Curvas ROC (Receiver Operating Characteristic) Comparativas')
plt.xlabel('Taxa de Falsos Positivos (1 - Especificidade)')
plt.ylabel('Taxa de Verdadeiros Positivos (Sensibilidade)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.4)
plt.show()

# B. Curvas Precision-Recall (PR)
plt.figure(figsize=(8, 7))
baseline_pr = np.sum(y_sample) / len(y_sample)
for name, p in probs.items():
    prec, rec, _ = precision_recall_curve(y_sample, p)
    plt.plot(rec, prec, lw=2, label=f'{name}')
plt.axhline(y=baseline_pr, color='#ff007f', linestyle='--', label=f'Baseline Proporcional ({baseline_pr*100:.1f}%)')
plt.title('Curvas Precision-Recall (PR) Comparativas')
plt.xlabel('Recall (Sensibilidade)')
plt.ylabel('Precision (Precisão Preditiva)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.4)
plt.show()

# C. Reliability Diagram (Curvas de Calibração)
plt.figure(figsize=(8, 7))
plt.plot([0, 1], [0, 1], 'k:', color='#e2e2e9', label='Calibração Perfeita')
for name, p in probs.items():
    fraction_of_positives, mean_predicted_value = calibration_curve(y_sample, p, n_bins=10)
    plt.plot(mean_predicted_value, fraction_of_positives, 's-', label=name, lw=2, markersize=4)
plt.title('Reliability Diagram: Curvas de Calibração de Probabilidade')
plt.xlabel('Probabilidade Prevista Média')
plt.ylabel('Proporção Real de Acidentes Graves')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.4)
plt.show()

# D. Curvas de Lift e Ganho Acumulado
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
n_samples = len(y_sample)
n_positives = np.sum(y_sample)
percentiles = np.linspace(0, 100, 101)
for name, p in probs.items():
    sorted_indices = np.argsort(p)[::-1]
    y_true_sorted = y_sample.iloc[sorted_indices].to_numpy()
    cum_gains = np.cumsum(y_true_sorted) / n_positives * 100
    cum_gains = np.insert(cum_gains, 0, 0)
    x_gains = np.linspace(0, 100, len(cum_gains))
    gains_interp = np.interp(percentiles, x_gains, cum_gains)
    ax1.plot(percentiles, gains_interp, label=name, lw=2)
    lift_interp = np.zeros_like(percentiles)
    lift_interp[1:] = (gains_interp[1:] / 100) / (percentiles[1:] / 100)
    lift_interp[0] = lift_interp[1]
    ax2.plot(percentiles, lift_interp, label=name, lw=2)
ax1.plot([0, 100], [0, 100], 'k--', color='#e2e2e9', label='Modelo Aleatório')
ax1.set_title('Curva de Ganho Acumulado (Cumulative Gain)')
ax1.set_xlabel('% da População Ordenada por Probabilidade')
ax1.set_ylabel('% de Casos Graves Detectados')
ax1.legend()
ax1.grid(True, linestyle=':', alpha=0.6)
ax2.axhline(y=1.0, color='r', linestyle='--', label='Lift = 1')
ax2.set_title('Curva de Lift')
ax2.set_xlabel('% da População Ordenada por Probabilidade')
ax2.set_ylabel('Lift (Ganho sobre o Acaso)')
ax2.legend()
ax2.grid(True, linestyle=':', alpha=0.6)
plt.show()

# E. Projeção Espacial 2D t-SNE (Separação de Classes)
print("Calculando projeção t-SNE de 1.200 amostras...")
tsne_indices = np.random.choice(len(X_sample_scaled), size=min(1200, len(X_sample_scaled)), replace=False)
X_tsne_input = X_sample_scaled[tsne_indices]
y_tsne_input = y_sample.iloc[tsne_indices]
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=600)
X_tsne_2d = tsne.fit_transform(X_tsne_input)

plt.figure(figsize=(9, 7))
plt.scatter(X_tsne_2d[y_tsne_input == 0, 0], X_tsne_2d[y_tsne_input == 0, 1], 
            color='#00f0ff', alpha=0.6, label='Leve/Moderado (0)', s=20)
plt.scatter(X_tsne_2d[y_tsne_input == 1, 0], X_tsne_2d[y_tsne_input == 1, 1], 
            color='#ff007f', alpha=0.6, label='Grave/Fatal (1)', s=20)
plt.title('Projeção Espacial 2D via t-SNE: Separação de Classes de Severidade')
plt.xlabel('t-SNE Componente 1')
plt.ylabel('t-SNE Componente 2')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.3)
plt.show()

# F. SHAP Summary Plot
print("Calculando contribuições SHAP de 500 amostras para o XGBoost...")
xgb_model = models['XGBoost']
explainer = shap.TreeExplainer(xgb_model)
shap_indices = np.random.choice(len(X_sample_scaled), size=500, replace=False)
X_shap_input = X_sample_scaled[shap_indices]
shap_values = explainer.shap_values(X_shap_input)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sample_df.iloc[shap_indices], feature_names=feature_cols, show=False)
plt.title('Explicação Global SHAP (Tree SHAP no XGBoost)', pad=20)
plt.show()
